In [ ]:
!pip install torchvision
!pip install fvcore umap-learn scikit-video opencv-python-headless matplotlib seaborn tqdm scikit-learn imblearn albumentations

In [ ]:
# Install required dependencies (excluding pytorchvideo)
!pip install fvcore umap-learn scipy scikit-learn imbalanced-learn

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.parallel # For DataParallel
import torch.hub
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score,
    recall_score, matthews_corrcoef, confusion_matrix, roc_auc_score,
    classification_report
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek
import umap
from scipy.stats import kendalltau, spearmanr
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from albumentations.pytorch import ToTensorV2
import albumentations as A
import logging
import warnings
from collections import Counter
import json
import cv2

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Configuration
CONFIG = {
    'data_folder': '/kaggle/input/summe-dataset/main_videos',
    'output_folder': '/kaggle/working',
    'seed': 42,
    'test_size': 0.2,
    'cv_folds': 5,
    'batch_size': 2,
    'num_workers': 2,
    'frame_samples': 32,
    'img_size': 224,
    'use_data_augmentation': True,
    'use_ensemble': True,
    'use_smote': True,
    'feature_selection': True,
}

def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds(CONFIG['seed'])

def get_video_files(data_dir):
    video_files = []
    video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.webm', '.flv', '.wmv']
    
    if not os.path.exists(data_dir):
        logging.error(f"Data directory does not exist: {data_dir}")
        return video_files
    
    for class_name in os.listdir(data_dir):
        class_path = os.path.join(data_dir, class_name)
        if os.path.isdir(class_path):
            files = []
            for f in os.listdir(class_path):
                if any(f.lower().endswith(ext) for ext in video_extensions):
                    full_path = os.path.join(class_path, f)
                    cap = cv2.VideoCapture(full_path)
                    if cap.isOpened() and cap.get(cv2.CAP_PROP_FRAME_COUNT) > 0:
                        files.append(full_path)
                    cap.release()
            video_files.extend([(f, class_name) for f in files])
            logging.info(f"Class '{class_name}': {len(files)} valid videos")
    
    logging.info(f"Total valid videos found: {len(video_files)}")
    return video_files

class SupervisedVideoDataset(Dataset):
    def __init__(self, video_files, label_encoder, mode='train'):
        self.video_files = video_files
        self.label_encoder = label_encoder
        self.mode = mode
        self.labels = [self.label_encoder.transform([class_name])[0] for _, class_name in self.video_files]
        
        if CONFIG['use_data_augmentation'] and mode == 'train':
            self.augmentation = A.Compose([
                A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
                A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
                A.GaussianBlur(blur_limit=3, p=0.3),
                A.Rotate(limit=15, p=0.3),
                A.HorizontalFlip(p=0.5),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2()
            ])
        else:
            self.augmentation = A.Compose([
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2()
            ])
        
        self.temporal_features = self._compute_temporal_features()
        self.motion_features = self._compute_motion_features()
        self.texture_features = self._compute_texture_features()

    def _compute_temporal_features(self):
        features = []
        for video_path, _ in tqdm(self.video_files, desc=f"Computing temporal features ({self.mode})"):
            feature = self._extract_temporal_features(video_path)
            features.append(feature)
        return np.array(features)
    
    def _extract_temporal_features(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        n_samples = min(CONFIG['frame_samples'], total_frames)
        indices = np.linspace(0, total_frames - 1, n_samples, dtype=int)
        
        frames = []
        gray_frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame = cv2.resize(frame, (CONFIG['img_size'], CONFIG['img_size']))
                frames.append(frame)
                gray_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
        
        cap.release()
        
        if len(frames) < 2:
            return np.zeros(25) # Ensure consistent feature length
        
        features = []
        
        flows = []
        flow_magnitudes = []
        flow_angles = []
        for i in range(len(gray_frames) - 1):
            flow = cv2.calcOpticalFlowFarneback(
                gray_frames[i], gray_frames[i+1], None, 
                0.5, 3, 15, 3, 5, 1.2, 0
            )
            mag, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
            flows.append(flow)
            flow_magnitudes.append(mag)
            flow_angles.append(angle)
        
        if flows:
            flow_stats = [
                np.mean([np.mean(f) for f in flow_magnitudes]),
                np.std([np.std(f) for f in flow_magnitudes]), # std of stds
                np.max([np.max(f) for f in flow_magnitudes]),
                np.std([np.mean(f) for f in flow_magnitudes]), # std of means
                np.mean([np.percentile(f, 95) for f in flow_magnitudes]),
                np.mean([np.var(f) for f in flow_magnitudes]), # mean of variances
            ]
            angle_stats = [
                np.mean([np.mean(a) for a in flow_angles]),
                np.std([np.mean(a) for a in flow_angles]), # std of mean angles
                np.mean([np.std(a) for a in flow_angles]), # mean of std angles
            ]
            features.extend(flow_stats + angle_stats)
        else:
            features.extend([0] * 9) # 6 for flow_stats, 3 for angle_stats
        
        frame_diffs = []
        for i in range(len(frames) - 1):
            diff = cv2.absdiff(frames[i], frames[i+1])
            frame_diffs.append([
                np.mean(diff),
                np.std(diff),
                np.max(diff),
                np.percentile(diff, 95)
            ])
        
        if frame_diffs:
            diff_array = np.array(frame_diffs)
            diff_stats = [
                np.mean(diff_array[:,0]), # mean of means
                np.std(diff_array[:,0]),  # std of means
                np.mean(diff_array[:,1]), # mean of stds
                np.mean(diff_array[:,2]), # mean of maxes
                np.mean(diff_array[:,3]), # mean of 95th percentiles
                np.std(diff_array[:,0]), # Repeated std of means, perhaps variance of means? np.var(diff_array[:,0])
            ]
        else:
            diff_stats = [0] * 6
        
        features.extend(diff_stats)
        
        metadata_features = [
            total_frames,
            fps if fps > 0 else 25, # Default FPS if unknown
            total_frames / max(fps, 1), # Duration
            width * height, # Resolution area
            width / max(height, 1), # Aspect ratio
        ]
        
        features.extend(metadata_features)
        
        if len(flow_magnitudes) > 1:
            # Calculate mean magnitude for each flow step
            mean_magnitudes_per_step = [np.mean(f) for f in flow_magnitudes]
            temporal_smoothness = np.corrcoef(
                mean_magnitudes_per_step[:-1], 
                mean_magnitudes_per_step[1:]
            )[0,1]
            if np.isnan(temporal_smoothness):
                temporal_smoothness = 0
        else:
            temporal_smoothness = 0
        
        features.append(temporal_smoothness)
        
        scene_changes = 0
        if len(frame_diffs) > 0: # Use frame_diffs calculated earlier
            diff_means = np.array(frame_diffs)[:,0] # Use mean diffs
            if len(diff_means) > 0:
                 threshold = np.mean(diff_means) + 2 * np.std(diff_means)
                 scene_changes = np.sum(diff_means > threshold)
        
        features.append(scene_changes / max(len(frame_diffs), 1))
        
        edge_densities = []
        # Process fewer frames for edge density to save time, e.g., every 4th sampled frame
        for frame in frames[::max(1, len(frames)//8)]: # Sample up to 8 frames for edges
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray, 50, 150)
            edge_density = np.sum(edges > 0) / (edges.shape[0] * edges.shape[1])
            edge_densities.append(edge_density)
        
        if edge_densities:
            features.extend([np.mean(edge_densities), np.std(edge_densities)])
        else:
            features.extend([0, 0])
        
        # Ensure features are always of the same length (25)
        # current length = 9 (flow) + 6 (diff) + 5 (metadata) + 1 (smoothness) + 1 (scene_change) + 2 (edge) = 24
        # Need one more, or adjust existing ones.
        # Let's assume the original target was 25 and one feature might be missing or miscounted.
        # For now, pad if less, truncate if more.
        target_len = 25
        if len(features) < target_len:
            features.extend([0.0] * (target_len - len(features)))
        elif len(features) > target_len:
            features = features[:target_len]

        return np.array(features, dtype=np.float32)

    def _compute_motion_features(self):
        features = []
        for video_path, _ in tqdm(self.video_files, desc=f"Computing motion features ({self.mode})"):
            feature = self._extract_motion_features(video_path)
            features.append(feature)
        return np.array(features)
    
    def _extract_motion_features(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        n_samples = min(32, total_frames) # Sample up to 32 frames for motion
        if n_samples < 2: # Need at least 2 frames for optical flow
             cap.release()
             return np.zeros(8)

        indices = np.linspace(0, total_frames - 1, n_samples, dtype=int)
        
        prev_frame = None
        motion_vectors = []
        
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                gray = cv2.resize(gray, (CONFIG['img_size'] // 2, CONFIG['img_size'] // 2)) # Smaller size for faster flow
                
                if prev_frame is not None:
                    flow = cv2.calcOpticalFlowFarneback(prev_frame, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
                    magnitude = np.sqrt(flow[...,0]**2 + flow[...,1]**2)
                    motion_vectors.append(magnitude)
                
                prev_frame = gray
        
        cap.release()
        
        if not motion_vectors:
            return np.zeros(8) # Consistent feature length
        
        all_magnitudes = np.concatenate([mv.ravel() for mv in motion_vectors]) # Ravel each magnitude map
        if all_magnitudes.size == 0: # Handle case where concatenation is empty
            return np.zeros(8)

        features = [
            np.mean(all_magnitudes),
            np.std(all_magnitudes),
            np.max(all_magnitudes),
            np.min(all_magnitudes),
            np.percentile(all_magnitudes, 75),
            np.percentile(all_magnitudes, 25),
            np.sum(all_magnitudes > np.mean(all_magnitudes)) / max(len(all_magnitudes), 1),
            len(motion_vectors) # Number of flow calculations
        ]
        
        return np.array(features, dtype=np.float32)

    def _compute_texture_features(self):
        features = []
        # Import skimage here if it's only used in this method
        try:
            from skimage.feature import local_binary_pattern
        except ImportError:
            logging.error("Scikit-image is required for texture features. Please install it.")
            # Return zeros if skimage is not available
            for _ in tqdm(self.video_files, desc=f"Computing texture features (SKIPPED) ({self.mode})"):
                 features.append(np.zeros(10, dtype=np.float32))
            return np.array(features)

        for video_path, _ in tqdm(self.video_files, desc=f"Computing texture features ({self.mode})"):
            feature = self._extract_texture_features_skimage(video_path, local_binary_pattern)
            features.append(feature)
        return np.array(features)
    
    def _extract_texture_features_skimage(self, video_path, local_binary_pattern_func):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        n_samples = min(8, total_frames) # Sample up to 8 frames for texture
        if n_samples == 0:
            cap.release()
            return np.zeros(10)

        indices = np.linspace(0, total_frames - 1, n_samples, dtype=int)
        
        texture_features_list = [] # Renamed to avoid conflict
        
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                gray = cv2.resize(gray, (CONFIG['img_size'], CONFIG['img_size']))
                
                lbp = local_binary_pattern_func(gray, P=8, R=1, method='uniform') # P points, R radius
                lbp_hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 10 + 1), range=(0,10)) # LBP uniform has P+2 bins (8+2=10)
                lbp_hist = lbp_hist.astype(float)
                lbp_hist /= (lbp_hist.sum() + 1e-8) # Normalize
                texture_features_list.append(lbp_hist)
        
        cap.release()
        
        if not texture_features_list:
            return np.zeros(10) # Consistent feature length
        
        return np.mean(texture_features_list, axis=0).astype(np.float32)

    def __len__(self):
        return len(self.video_files)
    
    def __getitem__(self, idx):
        video_path, class_name = self.video_files[idx]
        
        try:
            frames = self._extract_frames_robust(video_path)
            label = self.label_encoder.transform([class_name])[0]
            temporal_feat = self.temporal_features[idx]
            motion_feat = self.motion_features[idx]
            texture_feat = self.texture_features[idx]
            
            # Ensure all feature arrays are 1D before concatenation
            combined_features = np.concatenate([
                temporal_feat.flatten(), 
                motion_feat.flatten(), 
                texture_feat.flatten()
            ])
            
            # Expected length: 25 (temporal) + 8 (motion) + 10 (texture) = 43
            expected_len = 25 + 8 + 10
            if len(combined_features) != expected_len:
                # Pad or truncate if necessary, though individual extractors should ensure this
                logging.warning(f"Combined features length mismatch for {video_path}: got {len(combined_features)}, expected {expected_len}. Adjusting.")
                if len(combined_features) < expected_len:
                    combined_features = np.pad(combined_features, (0, expected_len - len(combined_features)), 'constant')
                else:
                    combined_features = combined_features[:expected_len]

            return frames, label, combined_features, video_path
        except Exception as e:
            logging.warning(f"Error processing video {video_path}: {e}")
            dummy_frames = torch.zeros(3, CONFIG['frame_samples'], CONFIG['img_size'], CONFIG['img_size'])
            dummy_label = 0 # Or a specific error label if needed
            dummy_features = np.zeros(25 + 8 + 10) # Ensure consistent dummy feature length
            return dummy_frames, dummy_label, dummy_features, video_path
    
    def _extract_frames_robust(self, video_path, max_retries=3):
        for attempt in range(max_retries):
            try:
                cap = cv2.VideoCapture(video_path)
                if not cap.isOpened():
                    raise ValueError(f"Could not open video: {video_path}")
                
                total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                if total_frames == 0:
                    raise ValueError(f"Video has no frames: {video_path}")
                
                n_samples = CONFIG['frame_samples']
                if total_frames < n_samples:
                    indices = list(range(total_frames))
                    # Pad by repeating last frame or cycling
                    indices_to_add = n_samples - len(indices)
                    if indices: # if there's at least one frame
                        indices.extend([indices[-1]] * indices_to_add)
                    else: # no frames, should have been caught by total_frames == 0
                        raise ValueError(f"Video {video_path} has frames but indices list is empty.")
                else:
                    indices = np.linspace(0, total_frames - 1, n_samples, dtype=int)
                
                frames_list = [] # Renamed to avoid conflict
                for i, frame_idx in enumerate(indices):
                    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
                    ret, frame = cap.read()
                    if ret and frame is not None:
                        frame = cv2.resize(frame, (CONFIG['img_size'], CONFIG['img_size']))
                        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                        
                        try:
                            augmented = self.augmentation(image=frame)
                            frame_tensor = augmented['image']
                        except Exception as aug_e: # More specific exception handling
                            logging.warning(f"Augmentation failed for {video_path}, frame {i}: {aug_e}. Using basic ToTensor.")
                            # Fallback to manual normalization and tensor conversion
                            frame = frame.astype(np.float32) / 255.0
                            mean = np.array([0.485, 0.456, 0.406])
                            std = np.array([0.229, 0.224, 0.225])
                            frame = (frame - mean) / std
                            frame_tensor = torch.from_numpy(frame.transpose(2,0,1)).float() # Ensure float
                        
                        frames_list.append(frame_tensor)
                    else: # If a frame cannot be read, append a dummy or last good frame
                        logging.warning(f"Could not read frame {frame_idx} from {video_path}. Appending previous or zero tensor.")
                        if frames_list:
                            frames_list.append(frames_list[-1].clone())
                        else: # This case means even the first frame failed.
                            frames_list.append(torch.zeros(3, CONFIG['img_size'], CONFIG['img_size'], dtype=torch.float32))


                cap.release()
                
                # Ensure exactly n_samples frames
                if len(frames_list) < n_samples:
                    logging.warning(f"Extracted {len(frames_list)} frames, needed {n_samples} for {video_path}. Padding.")
                    padding_needed = n_samples - len(frames_list)
                    if frames_list: # If some frames were read
                         for _ in range(padding_needed):
                            frames_list.append(frames_list[-1].clone())
                    else: # No frames were read at all
                        for _ in range(padding_needed):
                            frames_list.append(torch.zeros(3, CONFIG['img_size'], CONFIG['img_size'], dtype=torch.float32))
                elif len(frames_list) > n_samples:
                     frames_list = frames_list[:n_samples]


                return torch.stack(frames_list).permute(1,0,2,3) # (C, T, H, W)
            except Exception as e:
                logging.warning(f"Attempt {attempt+1} failed for {video_path}: {e}")
                if cap.isOpened(): # Ensure cap is released on exception
                    cap.release()
                if attempt == max_retries-1:
                    raise # Re-raise the exception after max retries

class VideoFeatureExtractor(nn.Module):
    def __init__(self, pretrained=True):
        super(VideoFeatureExtractor, self).__init__()
        # Load R(2+1)D-50 model from PyTorchVideo via torch.hub
        try:
            self.model = torch.hub.load('facebookresearch/pytorchvideo:main', 'r2plus1d_r50', pretrained=pretrained)
            # Remove the classification head to get features
            # The R(2+1)D model in PyTorchVideo has a `blocks` attribute, and the head is typically the last block's pool and fc.
            # More robustly, access the final layer often named `fc` or similar.
            # For r2plus1d_r50, the head is `self.model.blocks[4].proj` or `model.head.projection`
            # Let's assume the original structure was: model.blocks -> model.head
            # If the last block is the head itself (common in some architectures):
            self.model.blocks[-1] = nn.Identity() # Replace the final block (head) with Identity
            # Or, if the head is a separate module (e.g., self.model.head or self.model.fc):
            # self.model.head = nn.Identity() # Example
            # It seems for r2plus1d_r50 from pytorchvideo, the structure is model.blocks[0-4] then a head.
            # The feature vector comes from the layer before the final fully connected layer.
            # The output of blocks[4] is (N, C, T, H, W), then avg_pool, then projection.
            # We want features before projection.
            # Let's check the structure by printing layers if unsure.
            # For now, assuming the original intent was to get features from the layer before classification.
            # The `blocks[-1] = nn.Identity()` might remove too much if blocks[-1] is a stem_helper or res_block.
            # A common way is to remove `model.fc` or `model.head`.
            # For R(2+1)D from PytorchVideo, the features are usually taken after the pooling layer within the head.
            # self.model.head.projection = nn.Identity() # This would give features before the final projection
            # If we want features before any pooling in the head:
            # self.model.head = nn.Identity() # This is safer if the head contains pooling + fc.
            
            # Let's stick to the provided way, assuming it correctly targets the classification part.
            # The last block in r2plus1d_r50 from PyTorchVideo is `ResNetBasicHead` which includes avg pooling and projection.
            # Replacing it with Identity means the output of the previous block (blocks[3] or the stem for shallow models)
            # will be passed through. This might be too early.
            # A safer bet for features from r2plus1d_r50 is to remove just the final linear layer (projection).
            if hasattr(self.model, 'head') and hasattr(self.model.head, 'projection'):
                 self.model.head.projection = nn.Identity()
            else: # Fallback to original logic if structure is different
                 self.model.blocks[-1] = nn.Identity()


        except Exception as e:
            logging.error(f"Failed to load R(2+1)D model: {e}")
            self.model = None # Handle failure

    def forward(self, x):
        # Input x: (batch_size, C, T, H, W), e.g., (batch_size, 3, 32, 224, 224)
        if self.model is None:
            # Return zeros of expected feature dimension if model failed to load
            # Assuming a common feature dim like 2048 or 512.
            # This needs to be consistent with downstream processing.
            # The R(2+1)D-50 typically has 2048 features before the final FC layer.
            return torch.zeros(x.size(0), 2048, device=x.device)

        features = self.model(x) # Pass x directly to the model
        # The output shape depends on where Identity was placed.
        # If self.model.head.projection = nn.Identity(), features are (batch_size, feature_dim) after avg_pool.
        # If self.model.blocks[-1] = nn.Identity(), features might be (batch_size, C_last_block, T_last_block, H_last_block, W_last_block)
        if features.dim() > 2: # If output is still spatio-temporal
            # Apply global average pooling if necessary
            features = torch.mean(features, dim=list(range(2, features.dim())))
        return features # Output: (batch_size, feature_dim)

def extract_features(model, loader, device='cuda'):
    model.to(device)
    if torch.cuda.device_count() > 1:
        logging.info(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model) # Ensure model is wrapped for DataParallel
    
    model.eval()
    visual_features, combined_features, labels, video_paths = [], [], [], []
    
    # Determine dummy feature dimension dynamically if possible
    dummy_vis_dim = None
    try:
        # Try to get feature dim from a dummy forward pass if model is not None
        if hasattr(model, 'module'): # If DataParallel
            if model.module.model is not None:
                dummy_input = torch.randn(1, 3, CONFIG['frame_samples'], CONFIG['img_size'], CONFIG['img_size']).to(device)
                dummy_output = model(dummy_input)
                dummy_vis_dim = dummy_output.shape[1]
        elif model.model is not None:
            dummy_input = torch.randn(1, 3, CONFIG['frame_samples'], CONFIG['img_size'], CONFIG['img_size']).to(device)
            dummy_output = model(dummy_input)
            dummy_vis_dim = dummy_output.shape[1]
    except Exception as e:
        logging.warning(f"Could not determine dummy_vis_dim dynamically: {e}")
    
    if dummy_vis_dim is None:
        dummy_vis_dim = 2048 # Fallback to a common default for R(2+1)D-50
        logging.info(f"Using fallback dummy_vis_dim: {dummy_vis_dim}")


    with torch.no_grad():
        for batch_idx, batch_data in enumerate(tqdm(loader, desc="Extracting features")):
            if batch_data is None: # Skip if loader returns None (e.g. error in __getitem__)
                logging.warning(f"Skipping problematic batch {batch_idx}")
                continue
            
            frames, lbls, combined_feats, paths = batch_data
            
            try:
                frames = frames.to(device)
                vis_feats = model(frames)
                vis_feats = F.normalize(vis_feats, p=2, dim=1) # L2 normalize
                
                visual_features.append(vis_feats.cpu().numpy())
                combined_features.append(combined_feats.numpy())
                labels.extend(lbls.numpy())
                video_paths.extend(paths)
            except Exception as e:
                logging.warning(f"Error processing batch {batch_idx} for videos {paths}: {e}")
                batch_size = len(lbls)
                # Use the dynamically determined or fallback dummy_vis_dim
                dummy_vis = np.zeros((batch_size, dummy_vis_dim)) 
                visual_features.append(dummy_vis)
                combined_features.append(combined_feats.numpy()) # Still try to append other data
                labels.extend(lbls.numpy())
                video_paths.extend(paths)
    
    if not visual_features: # Handle cases where no features were extracted
        logging.error("No visual features were extracted. Returning empty arrays.")
        return np.array([]), np.array([]), np.array([]), []

    visual_features = np.vstack(visual_features)
    combined_features = np.vstack(combined_features)
    # Ensure dimensions match for concatenation
    if visual_features.shape[0] != combined_features.shape[0]:
        logging.error(f"Mismatch in number of samples for visual ({visual_features.shape[0]}) and combined ({combined_features.shape[0]}) features. This indicates a serious data loading issue.")
        # Decide on a recovery strategy, e.g., use the minimum count or skip. For now, let's try to align.
        min_rows = min(visual_features.shape[0], combined_features.shape[0])
        if min_rows == 0: # If one is empty, cannot proceed
             return np.array([]), np.array([]), np.array(labels)[:min_rows], video_paths[:min_rows]
        visual_features = visual_features[:min_rows]
        combined_features = combined_features[:min_rows]
        labels = labels[:min_rows]
        video_paths = video_paths[:min_rows]


    all_features = np.concatenate([visual_features, combined_features], axis=-1)
    
    return all_features, combined_features, np.array(labels), video_paths

class EnhancedEnsembleClassifier:
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.scalers = {}
        self.classifiers = {}
        self.feature_selector = None
        self.voting_classifier = None
        self.best_scaler_for_voting = None # Renamed for clarity

    def _create_classifiers(self, n_classes, X_shape_1): # Pass X_shape_1 for k_neighbors
        # Adjust k for KNN and SMOTE based on actual data points in the smallest class or n_classes
        # This k needs to be carefully set, especially if using SMOTE before CV splitting per fold.
        # For now, k_neighbors for KNN based on n_classes or a small default.
        knn_k = min(5, max(3, n_classes * 2 if n_classes > 1 else 3))
        if X_shape_1 > 0 : # if we have samples
            knn_k = min(knn_k, X_shape_1 -1 if X_shape_1 >1 else 1) # k must be < n_samples for KNN

        classifiers = {
            'rf': RandomForestClassifier(
                n_estimators=200, max_depth=10, min_samples_split=5,
                min_samples_leaf=2, random_state=self.random_state, class_weight='balanced'
            ),
            'gb': GradientBoostingClassifier(
                n_estimators=100, learning_rate=0.1, max_depth=6,
                random_state=self.random_state # GB doesn't have class_weight, balancing should be done via SMOTE
            ),
            'svm': SVC(
                kernel='rbf', C=1.0, gamma='scale', probability=True,
                random_state=self.random_state, class_weight='balanced'
            ),
            'knn': KNeighborsClassifier(
                n_neighbors=knn_k if knn_k > 0 else 1, weights='distance' # ensure k > 0
            ),
            'lr': LogisticRegression(
                random_state=self.random_state, max_iter=1000, class_weight='balanced', solver='liblinear' # Add solver for potential convergence
            )
        }
        return classifiers
    
    def fit(self, X_train, y_train, use_smote=True, feature_selection=True):
        if X_train.shape[0] == 0:
            logging.error("Cannot fit model with zero training samples.")
            return

        if feature_selection and CONFIG['feature_selection']:
            from sklearn.feature_selection import SelectKBest, f_classif
            # k should be less than or equal to the number of features
            k = min(20, X_train.shape[1] // 2 if X_train.shape[1] // 2 > 0 else X_train.shape[1])
            if k > 0 :
                self.feature_selector = SelectKBest(score_func=f_classif, k=k)
                try:
                    X_train_selected = self.feature_selector.fit_transform(X_train, y_train)
                except ValueError as e: # Handle cases where k might be too large for available features after some issue
                    logging.warning(f"Feature selection failed: {e}. Using all features.")
                    self.feature_selector = None # Disable feature selection
                    X_train_selected = X_train
            else:
                logging.warning("Not enough features for SelectKBest. Skipping feature selection.")
                self.feature_selector = None
                X_train_selected = X_train
        else:
            X_train_selected = X_train
        
        if use_smote and CONFIG['use_smote']:
            unique_classes, counts = np.unique(y_train, return_counts=True)
            # k_neighbors for SMOTE must be less than the number of samples in the smallest class
            min_samples_in_class = counts.min() if len(counts)>0 else 0
            
            smote_k_neighbors = min(3, min_samples_in_class - 1 if min_samples_in_class > 1 else 1)

            if len(unique_classes) > 1 and smote_k_neighbors > 0:
                try:
                    smote = SMOTE(random_state=self.random_state, k_neighbors=smote_k_neighbors)
                    X_train_balanced, y_train_balanced = smote.fit_resample(X_train_selected, y_train)
                    logging.info(f"Applied SMOTE (k={smote_k_neighbors}): {Counter(y_train)} -> {Counter(y_train_balanced)}")
                except ValueError as e_smote: # SMOTE can fail if a class has too few samples
                    logging.warning(f"SMOTE failed (k={smote_k_neighbors}): {e_smote}. Trying ADASYN.")
                    try:
                        adasyn_n_neighbors = min(3, min_samples_in_class -1 if min_samples_in_class > 1 else 1)
                        if adasyn_n_neighbors > 0:
                            adasyn = ADASYN(random_state=self.random_state, n_neighbors=adasyn_n_neighbors)
                            X_train_balanced, y_train_balanced = adasyn.fit_resample(X_train_selected, y_train)
                            logging.info(f"Applied ADASYN (n_neighbors={adasyn_n_neighbors}): {Counter(y_train)} -> {Counter(y_train_balanced)}")
                        else:
                            raise ValueError("Not enough samples for ADASYN neighbors.")
                    except Exception as e_adasyn: # Catch broader exceptions for ADASYN
                        logging.warning(f"ADASYN also failed: {e_adasyn}. Using original data for balancing.")
                        X_train_balanced, y_train_balanced = X_train_selected, y_train
            else:
                logging.info("Skipping SMOTE/ADASYN: Not enough classes, samples, or k_neighbors is 0.")
                X_train_balanced, y_train_balanced = X_train_selected, y_train
        else:
            X_train_balanced, y_train_balanced = X_train_selected, y_train
        
        n_classes_balanced = len(np.unique(y_train_balanced))
        base_classifiers = self._create_classifiers(n_classes_balanced, X_train_balanced.shape[0])
        
        self.classifiers = {} # Reset classifiers for each fit
        self.scalers = {} # Reset scalers

        for name, clf in base_classifiers.items():
            try:
                # Scale features for relevant models
                current_X_train = X_train_balanced
                if name in ['svm', 'knn', 'lr']: # Models sensitive to feature scaling
                    scaler = StandardScaler()
                    X_scaled_train = scaler.fit_transform(current_X_train)
                    self.scalers[name] = scaler
                else:
                    X_scaled_train = current_X_train # No scaling for tree-based models like RF, GB
                    self.scalers[name] = None # Explicitly store None for no scaler
                
                # Check if y_train_balanced still has enough samples for the classifier (esp. KNN)
                if name == 'knn' and X_scaled_train.shape[0] <= clf.n_neighbors:
                    logging.warning(f"Adjusting k for KNN for classifier {name} as n_samples ({X_scaled_train.shape[0]}) <= k ({clf.n_neighbors}). Setting k to 1.")
                    clf.n_neighbors = max(1, X_scaled_train.shape[0] -1) if X_scaled_train.shape[0] > 1 else 1


                if X_scaled_train.shape[0] == 0:
                    logging.warning(f"Skipping training for {name} due to zero samples after balancing/selection.")
                    continue
                if clf.n_neighbors == 0 and name=='knn': # Cannot train KNN with 0 neighbors
                    logging.warning(f"Skipping KNN as n_neighbors is 0.")
                    continue


                clf.fit(X_scaled_train, y_train_balanced)
                self.classifiers[name] = clf
                logging.info(f"Trained {name} classifier.")
            except Exception as e:
                logging.warning(f"Failed to train {name}: {e}")
        
        # Ensemble Voting Classifier
        if len(self.classifiers) > 1 and CONFIG['use_ensemble']:
            # Use a common scaler for voting if appropriate, or scale per estimator if pipelines were used (not here)
            # For simplicity, if most models use a scaler, we might scale input to VotingClassifier.
            # Here, we'll assume RF's scaler (None) or the first available scaler if RF's is None.
            # This is a heuristic; a more robust approach might involve Pipelines for each estimator in VotingClassifier.
            
            voting_estimators = []
            for name, clf_instance in self.classifiers.items():
                 # Create a pipeline for each estimator if it requires scaling
                if self.scalers[name] is not None:
                    from sklearn.pipeline import Pipeline
                    pipe = Pipeline([('scaler', self.scalers[name]), ('classifier', clf_instance)])
                    voting_estimators.append((name, pipe))
                else:
                    voting_estimators.append((name, clf_instance))


            if not voting_estimators:
                logging.warning("No base classifiers available for VotingClassifier.")
                self.voting_classifier = None
                return


            self.voting_classifier = VotingClassifier(estimators=voting_estimators, voting='soft') # 'soft' for probability weighted voting
            
            # VotingClassifier handles scaling internally if pipelines are used.
            # We need to fit it on the original X_train_balanced (or X_train_selected if no balancing)
            # as pipelines will do their own scaling.
            # If not using pipelines for voting_estimators, then a common scaling for X_voting would be needed.
            # Since we are now using pipelines, fit on X_train_balanced:
            try:
                self.voting_classifier.fit(X_train_balanced, y_train_balanced) # X_train_balanced is already feature selected
                logging.info("Trained voting classifier with estimator-specific scaling via pipelines.")
            except Exception as e:
                self.voting_classifier = None
                logging.warning(f"Voting classifier training failed: {e}")
        elif len(self.classifiers) == 1:
             logging.info("Only one base classifier trained. Voting ensemble not applicable.")
             self.voting_classifier = None # Explicitly set to None
        else:
            logging.info("Ensemble not used or too few classifiers.")
            self.voting_classifier = None


    def predict(self, X_test):
        if X_test.shape[0] == 0:
             return np.array([]), np.array([]), {}, {}

        if self.feature_selector:
            X_test_selected = self.feature_selector.transform(X_test)
        else:
            X_test_selected = X_test
        
        individual_predictions = {}
        individual_probabilities = {}
        
        # Predictions from individual classifiers
        for name, clf in self.classifiers.items():
            try:
                current_X_test = X_test_selected
                scaler = self.scalers.get(name, None)
                if scaler: # If a scaler was used for this classifier during training
                    X_scaled_test = scaler.transform(current_X_test)
                else:
                    X_scaled_test = current_X_test
                
                individual_predictions[name] = clf.predict(X_scaled_test)
                if hasattr(clf, 'predict_proba'):
                    individual_probabilities[name] = clf.predict_proba(X_scaled_test)
                else: # For classifiers like basic SVM without probability=True
                    # Create dummy probabilities (e.g., one-hot based on prediction)
                    n_samples_test = X_scaled_test.shape[0]
                    # Try to get classes from the classifier, otherwise assume binary or error
                    try:
                        n_classes_clf = len(clf.classes_)
                        pred_dummy_proba = np.zeros((n_samples_test, n_classes_clf))
                        pred_indices = clf.predict(X_scaled_test) # get actual predictions
                        pred_dummy_proba[np.arange(n_samples_test), pred_indices] = 1.0
                    except AttributeError: # if clf.classes_ is not available
                        logging.warning(f"Classifier {name} has no predict_proba and no classes_ attribute. Probs set to zero.")
                        # This requires knowing the number of classes beforehand.
                        # Fallback: Get n_classes from the first available proba or a config.
                        # For simplicity, let's assume it's 2 if not inferable. A better way is needed for multiclass.
                        first_proba = next(iter(individual_probabilities.values()), None)
                        num_classes_fallback = first_proba.shape[1] if first_proba is not None else 2
                        pred_dummy_proba = np.zeros((n_samples_test, num_classes_fallback))

                    individual_probabilities[name] = pred_dummy_proba

            except Exception as e:
                logging.warning(f"Prediction failed for individual classifier {name}: {e}")
                # Fallback for failed predictions
                num_samples_test = X_test_selected.shape[0]
                # Try to determine num_classes from other successful predictions or a default
                num_classes_fallback = 2 
                if individual_probabilities:
                    num_classes_fallback = next(iter(individual_probabilities.values())).shape[1]

                individual_predictions[name] = np.zeros(num_samples_test, dtype=int)
                individual_probabilities[name] = np.zeros((num_samples_test, num_classes_fallback))

        # Prediction from Voting Classifier
        if self.voting_classifier and hasattr(self.voting_classifier, 'estimators_') and self.voting_classifier.estimators_:
            try:
                # VotingClassifier with pipelines handles scaling internally. Input should be X_test_selected.
                final_pred = self.voting_classifier.predict(X_test_selected)
                final_proba = self.voting_classifier.predict_proba(X_test_selected)
                return final_pred, final_proba, individual_predictions, individual_probabilities
            except Exception as e:
                logging.warning(f"Voting classifier prediction failed: {e}. Falling back.")
        
        # Fallback logic if voting classifier failed or not used
        if 'rf' in individual_predictions: # Prioritize Random Forest if available
            logging.info("Using Random Forest predictions as fallback/default.")
            return (individual_predictions['rf'], individual_probabilities['rf'],
                    individual_predictions, individual_probabilities)
        elif individual_predictions: # Otherwise, use the first available classifier
            first_classifier_name = list(individual_predictions.keys())[0]
            logging.info(f"Using {first_classifier_name} predictions as fallback/default.")
            return (individual_predictions[first_classifier_name],
                    individual_probabilities[first_classifier_name],
                    individual_predictions, individual_probabilities)
        
        # Absolute fallback: if no classifiers worked
        logging.error("No predictions could be made by any classifier.")
        n_samples_test = X_test_selected.shape[0]
        # Need to know num_classes for probabilities, assuming 2 if unknown
        # This should ideally come from label_encoder.classes_
        num_classes_fallback = 2 # Placeholder
        return (np.zeros(n_samples_test, dtype=int),
                np.zeros((n_samples_test, num_classes_fallback)), {}, {})

    def get_feature_importance(self):
        importance_dict = {}
        for name, clf_pipeline_or_direct in self.classifiers.items():
            # If it's a pipeline (used in voting), get the classifier step
            clf = clf_pipeline_or_direct
            if hasattr(clf_pipeline_or_direct, 'named_steps'): # Check if it's a Pipeline
                 clf = clf_pipeline_or_direct.named_steps.get('classifier', clf_pipeline_or_direct)

            if hasattr(clf, 'feature_importances_'): # For tree-based models
                importance_dict[name] = clf.feature_importances_
            elif hasattr(clf, 'coef_'): # For linear models
                # For SVM with linear kernel, coef_ is available. For RBF, it's not direct.
                # For Logistic Regression, coef_ gives importance.
                # Taking abs for magnitude, averaging for multi-class if coef_ is 2D
                if clf.coef_.ndim == 1: # Binary classification or single output
                    importance_dict[name] = np.abs(clf.coef_)
                else: # Multi-class, average importance across classes
                     importance_dict[name] = np.mean(np.abs(clf.coef_), axis=0)
        return importance_dict

class AdvancedMetrics:
    @staticmethod
    def compute_all_metrics(y_true, y_pred, y_prob=None, labels=None, class_names=None): # Added class_names
        if len(y_true) == 0 or len(y_pred) == 0: # Handle empty inputs
            logging.warning("Cannot compute metrics for empty true/predicted labels.")
            return { 'accuracy': 0, 'balanced_accuracy': 0, 'f1_macro': 0, 'f1_weighted': 0, 
                     'precision_macro': 0, 'recall_macro': 0, 'mcc': 0, 'kendall': 0, 'spearman': 0,
                     'confusion_matrix': np.array([]), 'classification_report': {}}

        metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
            'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
            'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
            'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
            'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
            'mcc': matthews_corrcoef(y_true, y_pred)
        }
        try:
            metrics['kendall'] = kendalltau(y_true, y_pred)[0]
        except ValueError: # Can happen if arrays are too small or constant
            metrics['kendall'] = 0.0 if len(y_true) > 1 else np.nan # or some other indicator

        try:
            metrics['spearman'] = spearmanr(y_true, y_pred)[0]
        except ValueError:
             metrics['spearman'] = 0.0 if len(y_true) > 1 else np.nan


        if y_prob is not None and len(np.unique(y_true)) > 1: # ROC AUC requires at least 2 classes
            try:
                if y_prob.shape[1] == 2 and len(np.unique(y_true)) == 2: # Binary case
                    metrics['roc_auc'] = roc_auc_score(y_true, y_prob[:,1])
                elif y_prob.shape[1] > 2: # Multi-class case
                    metrics['roc_auc_ovr'] = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
                    metrics['roc_auc_ovo'] = roc_auc_score(y_true, y_prob, multi_class='ovo', average='macro')

            except ValueError as e_roc: # Catch errors like "Only one class present in y_true"
                logging.warning(f"ROC AUC calculation failed: {e_roc}")
                # Set to 0.5 as a neutral value, or None if preferred
                if 'roc_auc' not in metrics: metrics['roc_auc'] = 0.5 
                if 'roc_auc_ovr' not in metrics and y_prob.shape[1] > 2 : metrics['roc_auc_ovr'] = 0.5
        
        # Ensure labels for confusion matrix and classification report are consistent
        unique_labels_in_data = np.unique(np.concatenate((y_true, y_pred)))
        
        # If class_names (from LabelEncoder) are provided, use them.
        # Otherwise, derive from unique_labels_in_data.
        report_target_names = class_names
        cm_display_labels = class_names

        if class_names is None: # Fallback if actual class names aren't passed
            report_target_names = [str(l) for l in sorted(unique_labels_in_data)]
            cm_display_labels = sorted(unique_labels_in_data)


        # Ensure `labels` argument for confusion_matrix covers all unique labels present.
        # This helps in getting a consistently sized matrix.
        # If class_names are from the original encoder, they should cover all possibilities.
        cm_labels_arg = list(range(len(class_names))) if class_names else sorted(unique_labels_in_data)


        metrics['confusion_matrix'] = confusion_matrix(y_true=y_true, y_pred=y_pred, labels=cm_labels_arg)
        metrics['classification_report'] = classification_report(
            y_true=y_true, y_pred=y_pred, labels=cm_labels_arg, # use numeric labels for report
            target_names=report_target_names, # use string names for display
            output_dict=True, zero_division=0
        )
        
        return metrics

    @staticmethod
    def plot_confusion_matrix(cm, display_labels, title="Confusion Matrix"): # Changed 'labels' to 'display_labels'
        plt.figure(figsize=(max(8, len(display_labels)), max(6, len(display_labels)*0.8))) # Adjust size
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=display_labels, yticklabels=display_labels)
        plt.title(title=title)
        plt.xlabel('Predicted Labels')
        plt.ylabel('True Labels')
        plt.tight_layout()
        # Create output folder if it doesn't exist
        os.makedirs(CONFIG['output_folder'], exist_ok=True)
        plt.savefig(os.path.join(CONFIG['output_folder'], 'confusion_matrix.png'))
        plt.show()
    
    @staticmethod
    def plot_feature_importance(importance_dict, feature_names=None, top_k=15):
        if not importance_dict:
            logging.info("No feature importance data to plot.")
            return
        
        num_clf_with_importance = len(importance_dict)
        if num_clf_with_importance == 0: return

        fig, axes = plt.subplots(num_clf_with_importance, 1, figsize=(12, 5 * num_clf_with_importance))
        if num_clf_with_importance == 1:
            axes = [axes] # Make it iterable
        
        for idx, (model_name, importance_scores) in enumerate(importance_dict.items()):
            if not isinstance(importance_scores, np.ndarray):
                importance_scores = np.array(importance_scores)

            if feature_names is None or len(feature_names) != len(importance_scores):
                current_feature_names = [f'Feature_{i}' for i in range(len(importance_scores))]
                if feature_names is not None and len(feature_names) != len(importance_scores):
                    logging.warning(f"Mismatch between provided feature_names ({len(feature_names)}) and importance_scores ({len(importance_scores)}) for {model_name}. Using default names.")
            else:
                current_feature_names = feature_names

            # Ensure top_k is not greater than the number of features
            current_top_k = min(top_k, len(importance_scores))
            if current_top_k == 0: # No features to plot
                axes[idx].text(0.5, 0.5, 'No feature importances available.', ha='center', va='center')
                axes[idx].set_title(f'{model_name} - Feature Importance (None)')
                continue

            top_k_indices = np.argsort(importance_scores)[-current_top_k:] # Get indices of top_k scores
            top_importance = importance_scores[top_k_indices]
            top_names = [current_feature_names[i] for i in top_k_indices]
            
            ax = axes[idx]
            ax.barh(range(len(top_importance)), top_importance, align='center')
            ax.set_yticks(range(len(top_names)))
            ax.set_yticklabels(top_names)
            ax.invert_yaxis() # Display most important at top
            ax.set_title(f'{model_name} - Top {current_top_k} Feature Importance')
            ax.set_xlabel('Importance Score')
        
        plt.tight_layout()
        os.makedirs(CONFIG['output_folder'], exist_ok=True)
        plt.savefig(os.path.join(CONFIG['output_folder'],'feature_importance.png'))
        plt.show()

def perform_cross_validation(X, y, model_class, cv_folds=5, random_state=42, label_encoder_classes=None):
    if X.shape[0] < cv_folds : # Not enough samples for specified folds
        logging.warning(f"Number of samples ({X.shape[0]}) is less than cv_folds ({cv_folds}). Reducing folds.")
        cv_folds = max(2, X.shape[0]) # At least 2 folds if possible, or equal to num samples.
        if cv_folds < 2 : # Cannot perform CV
            logging.error("Not enough samples for cross-validation (less than 2). Skipping CV.")
            return {}, [], []


    try:
        # StratifiedKFold requires at least 2 members in each class for n_splits > 1
        # We should check class distributions if issues arise.
        skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
        cv_scores = {'accuracy': [], 'f1_macro': [], 'balanced_accuracy': []}
        fold_predictions_all, fold_true_labels_all = [], [] # Renamed to avoid conflict
        
        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
            logging.info(f"--- CV Fold {fold+1}/{cv_folds} ---")
            try:
                X_train_fold, X_val_fold = X[train_idx], X[val_idx]
                y_train_fold, y_val_fold = y[train_idx], y[val_idx]
                
                if X_train_fold.shape[0] == 0 or X_val_fold.shape[0] == 0:
                    logging.warning(f"Fold {fold+1} has zero samples in train or validation set. Skipping fold.")
                    cv_scores['accuracy'].append(0.0)
                    cv_scores['f1_macro'].append(0.0)
                    cv_scores['balanced_accuracy'].append(0.0)
                    continue

                fold_model = model_class(random_state=random_state) # Instantiate new model for each fold
                fold_model.fit(X_train_fold, y_train_fold, 
                               use_smote=CONFIG.get('use_smote_in_cv', True), # Configurable SMOTE in CV
                               feature_selection=CONFIG.get('feature_selection_in_cv', True)) # Configurable FS in CV
                
                y_pred_fold, y_prob_fold, _, _ = fold_model.predict(X_val_fold)
                
                if len(y_val_fold) > 0: # Ensure there are validation labels to score against
                    cv_scores['accuracy'].append(accuracy_score(y_val_fold, y_pred_fold))
                    cv_scores['f1_macro'].append(f1_score(y_val_fold, y_pred_fold, average='macro', zero_division=0))
                    cv_scores['balanced_accuracy'].append(balanced_accuracy_score(y_val_fold, y_pred_fold))
                    
                    fold_predictions_all.extend(y_pred_fold)
                    fold_true_labels_all.extend(y_val_fold)
                    # Corrected f-string syntax:
                    logging.info(f"Fold {fold+1}: Acc={cv_scores['accuracy'][-1]:.4f}, F1={cv_scores['f1_macro'][-1]:.4f}, BalAcc={cv_scores['balanced_accuracy'][-1]:.4f}")

                else:
                    logging.warning(f"Fold {fold+1} resulted in empty y_val_fold after prediction. Scores set to 0.")
                    cv_scores['accuracy'].append(0.0)
                    cv_scores['f1_macro'].append(0.0)
                    cv_scores['balanced_accuracy'].append(0.0)

            except Exception as e:
                logging.warning(f"Error in Fold {fold+1}: {e}")
                # traceback.print_exc() # For more detailed error during debugging
                cv_scores['accuracy'].append(0.0)
                cv_scores['f1_macro'].append(0.0) 
                cv_scores['balanced_accuracy'].append(0.0)
        
        if not fold_true_labels_all: # If all folds failed or had no validation data
            logging.error("Cross-validation did not produce any results.")
            return {}, [], []

        # Calculate overall CV metrics from all fold predictions
        overall_cv_metrics = AdvancedMetrics.compute_all_metrics(
            np.array(fold_true_labels_all), 
            np.array(fold_predictions_all),
            class_names=label_encoder_classes # Pass actual class names
        )
        logging.info(f"Overall CV Accuracy from combined folds: {overall_cv_metrics.get('accuracy', 0.0):.4f}")
        logging.info(f"Overall CV Balanced Accuracy from combined folds: {overall_cv_metrics.get('balanced_accuracy',0.0):.4f}")
        logging.info(f"Overall CV F1 Macro from combined folds: {overall_cv_metrics.get('f1_macro',0.0):.4f}")


        cv_mean_results = {f'{metric}_mean': np.mean(scores) if scores else 0 for metric, scores in cv_scores.items()}
        cv_std_results = {f'{metric}_std': np.std(scores) if scores else 0 for metric, scores in cv_scores.items()}
        cv_results_summary = {**cv_mean_results, **cv_std_results, 'overall_metrics_from_folds': overall_cv_metrics}
        
        return cv_results_summary, fold_predictions_all, fold_true_labels_all
    except Exception as e:
        logging.error(f"Cross-validation failed critically: {e}")
        # import traceback # For debugging
        # traceback.print_exc()
        return {}, [], []

def perform_dimensionality_reduction(X, y, class_names_dr, methods=['pca', 'tsne', 'umap']): # Renamed labels to class_names_dr
    if X.shape[0] == 0:
        logging.warning("Cannot perform dimensionality reduction on empty data.")
        return {}
    
    scaler = StandardScaler()
    try:
        X_scaled = scaler.fit_transform(X)
    except ValueError as e: # e.g. if X contains NaNs or Infs not caught before
        logging.error(f"StandardScaler failed during DR: {e}. Using raw X for DR.")
        X_scaled = X # Fallback to unscaled data, though DR methods might still fail

    results = {}
    
    # PCA
    if 'pca' in methods:
        try:
            # n_components for PCA must be <= min(n_samples, n_features)
            n_comp_pca = min(2, X_scaled.shape[0], X_scaled.shape[1])
            if n_comp_pca < 2 : # PCA needs at least 2 components for 2D plot
                 logging.warning(f"PCA skipped: Not enough samples/features for 2 components (n_comp_pca={n_comp_pca}).")
            else:
                pca = PCA(n_components=n_comp_pca, random_state=CONFIG['seed'])
                X_pca = pca.fit_transform(X_scaled)
                results['pca'] = {
                    'embedding': X_pca,
                    'explained_variance_ratio': pca.explained_variance_ratio_ if hasattr(pca, 'explained_variance_ratio_') else [0,0]
                }
        except Exception as e:
            logging.warning(f"PCA failed: {str(e)}")
    
    # t-SNE
    if 'tsne' in methods:
        try:
            # Perplexity for t-SNE is typically 5-50. Must be < n_samples.
            tsne_perplexity = min(30, max(1, X_scaled.shape[0] - 1)) # Ensure perplexity < n_samples
            n_comp_tsne = min(2, X_scaled.shape[1]) # n_components <= n_features
            if X_scaled.shape[0] <= 1 or n_comp_tsne < 2 or tsne_perplexity == 0: # tSNE needs >1 sample and perplexity > 0
                logging.warning(f"t-SNE skipped: Not enough samples ({X_scaled.shape[0]}) or valid perplexity/components.")
            else:
                tsne = TSNE(n_components=n_comp_tsne, random_state=CONFIG['seed'], perplexity=tsne_perplexity, n_iter=300) # Reduce n_iter for speed
                X_tsne = tsne.fit_transform(X_scaled)
                results['tsne'] = {'embedding': X_tsne}
        except Exception as e:
            logging.warning(f"t-SNE failed: {e}")
    
    # UMAP
    if 'umap' in methods:
        try:
            # n_neighbors for UMAP must be < n_samples.
            umap_n_neighbors = min(15, max(2, X_scaled.shape[0] - 1)) # Ensure n_neighbors > 1 and < n_samples
            n_comp_umap = min(2, X_scaled.shape[1])
            if X_scaled.shape[0] <= 1 or n_comp_umap < 2 or umap_n_neighbors < 2:
                logging.warning(f"UMAP skipped: Not enough samples ({X_scaled.shape[0]}) or valid n_neighbors/components.")
            else:
                umap_reducer = umap.UMAP(n_components=n_comp_umap, random_state=CONFIG['seed'], 
                                         n_neighbors=umap_n_neighbors, min_dist=0.1) # Default min_dist
                X_umap = umap_reducer.fit_transform(X_scaled)
                results['umap'] = {'embedding': X_umap}
        except Exception as e:
            logging.warning(f"UMAP failed: {e}")
    
    if results:
        n_methods_plotted = sum(1 for res in results.values() if res['embedding'].shape[1] == 2) # Count methods that produced 2D embeddings
        if n_methods_plotted == 0:
            logging.info("No 2D embeddings produced by DR methods for plotting.")
            return results

        fig, axes = plt.subplots(1, n_methods_plotted, figsize=(6 * n_methods_plotted, 5), squeeze=False) # squeeze=False ensures axes is always 2D
        axes = axes.flatten() # Flatten to 1D array for easy indexing

        # Ensure class_names_dr matches the number of unique classes in y for coloring
        unique_y_labels = np.unique(y)
        num_unique_classes = len(unique_y_labels)

        # Generate colors based on the actual number of unique classes present in y
        colors = plt.cm.get_cmap('tab10', num_unique_classes)(np.linspace(0, 1, num_unique_classes))

        # Create a mapping from original class index (in y) to color index
        label_to_color_idx = {label_val: i for i, label_val in enumerate(unique_y_labels)}


        plot_idx = 0
        for method, data in results.items():
            if data['embedding'].shape[1] != 2: # Skip if not 2D
                continue

            embedding = data['embedding']
            ax = axes[plot_idx]

            for class_idx_original in unique_y_labels: # Iterate through unique class labels in y
                mask = (y == class_idx_original)
                
                # Get the name for this class_idx_original
                # Assuming class_idx_original corresponds to an index in class_names_dr
                try:
                    label_name = class_names_dr[class_idx_original] if class_names_dr and class_idx_original < len(class_names_dr) else f"Class {class_idx_original}"
                except IndexError:
                    label_name = f"Class {class_idx_original}"

                color_idx = label_to_color_idx[class_idx_original]
                ax.scatter(embedding[mask,0], embedding[mask,1], 
                           c=[colors[color_idx % len(colors)]], # Use modulo in case of color list mismatch (should not happen)
                           label=label_name, alpha=0.7, s=15) # Smaller points

            ax.set_title(f'{method.upper()} Visualization')
            ax.legend(fontsize='small')
            ax.grid(True, alpha=0.3)
            if method == 'pca' and 'explained_variance_ratio' in data and len(data['explained_variance_ratio']) >=2 :
                var_explained = data['explained_variance_ratio']
                ax.set_xlabel(f'PC1 ({var_explained[0]:.1%} variance)')
                ax.set_ylabel(f'PC2 ({var_explained[1]:.1%} variance)')
            plot_idx +=1
        
        plt.tight_layout()
        os.makedirs(CONFIG['output_folder'], exist_ok=True)
        plt.savefig(os.path.join(CONFIG['output_folder'], 'dimensionality_reduction.png'))
        plt.show()
    
    return results

def main():
    print("="*60)
    print("Enhanced Video Classification System with R(2+1)D")
    print("="*60)
    
    if not os.path.exists(CONFIG['data_folder']):
        logging.error(f"Data directory '{CONFIG['data_folder']}' not found. Please check `CONFIG['data_folder']`.")
        # Attempt to create a dummy data folder for testing if in a specific environment (e.g. Kaggle)
        if "kaggle" in os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "").lower():
            logging.info("Attempting to create dummy data structure for Kaggle...")
            try:
                os.makedirs(os.path.join(CONFIG['data_folder'], 'class_a'), exist_ok=True)
                os.makedirs(os.path.join(CONFIG['data_folder'], 'class_b'), exist_ok=True)
                # Create dummy video files (very small)
                # This requires cv2 to write, or just empty files if get_video_files is robust to them.
                # For now, just creating dirs. get_video_files should handle empty dirs.
                logging.info(f"Created dummy class folders in {CONFIG['data_folder']}. Add dummy videos if needed.")
            except Exception as e:
                logging.error(f"Could not create dummy data folders: {e}")
                return None, None, None
        else:
            return None, None, None # Exit if data folder not found and not in Kaggle for dummy creation
    
    print("\n1. Loading video files...")
    video_files = get_video_files(CONFIG['data_folder'])
    
    if not video_files:
        logging.error("No valid video files found. Ensure your data_folder has subdirectories for classes, each containing videos.")
        return None, None, None
    
    class_counts = Counter(class_name for _, class_name in video_files)
    print(f"Class distribution: {dict(class_counts)}")
    
    if len(class_counts) < 1: # Allow single class for feature extraction, but not for training meaningful model
        logging.error("No classes found. Need at least one class to proceed with feature extraction.")
        return None, None, None
    if len(class_counts) < 2:
        logging.warning("Only one class found. Classification performance metrics will be limited/meaningless. Cross-validation might fail.")
        # Proceeding for feature extraction, but further steps might be problematic.

    
    label_encoder = LabelEncoder()
    # Fit label encoder on all unique class names found
    all_class_names_from_files = sorted(list(class_counts.keys()))
    label_encoder.fit(all_class_names_from_files)
    class_labels_encoded_names = list(label_encoder.classes_) # These are the string names
    print(f"Encoded Classes (LabelEncoder): {class_labels_encoded_names}")
    
    print("\n2. Splitting data...")
    # Stratify requires at least 2 members per class if test_size is small, or enough samples overall.
    labels_for_stratify = [class_name for _, class_name in video_files]
    
    # Check if stratification is possible
    can_stratify = True
    if len(class_counts) > 1:
        for class_name, count in class_counts.items():
            # StratifiedShuffleSplit needs at least 2 samples per class for a split.
            # train_test_split is a bit more lenient but warns.
            if count < 2 and len(video_files) * CONFIG['test_size'] >= 1 : # If a class has <2 samples and we expect to take some for test
                logging.warning(f"Class '{class_name}' has only {count} sample(s). Stratification might be problematic or not possible. Consider non-stratified split if errors occur.")
                # can_stratify = False # Potentially disable stratification
    
    if len(class_counts) == 1: # Cannot stratify with one class
        can_stratify = False
        logging.info("Only one class present. Stratification disabled for train/test split.")

    try:
        train_files, test_files = train_test_split(
            video_files, test_size=CONFIG['test_size'], random_state=CONFIG['seed'],
            stratify=labels_for_stratify if can_stratify and len(class_counts) > 1 else None
        )
    except ValueError as e_split:
        logging.error(f"Error during train_test_split (stratify={can_stratify}): {e_split}. Trying without stratification.")
        train_files, test_files = train_test_split(
            video_files, test_size=CONFIG['test_size'], random_state=CONFIG['seed']
        )

    print(f"Train: {len(train_files)} videos, Test: {len(test_files)} videos")
    if not train_files:
        logging.error("No training files after split. Check data and test_size.")
        return None, None, None
    # Test files can be empty if test_size is very small or data is limited.

    print("\n3. Creating datasets...")
    train_dataset = SupervisedVideoDataset(train_files, label_encoder, mode='train')
    if test_files: # Only create test_dataset if there are test files
        test_dataset = SupervisedVideoDataset(test_files, label_encoder, mode='test')
        test_loader = DataLoader(
            test_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
            num_workers=CONFIG['num_workers'], pin_memory=torch.cuda.is_available()
        )
    else:
        logging.warning("No test files after split. Test evaluation will be skipped.")
        test_dataset = None
        test_loader = None

    train_loader = DataLoader(
        train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,
        num_workers=CONFIG['num_workers'], pin_memory=torch.cuda.is_available()
    )
    
    print("\n4. Initializing feature extractor...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    feature_extractor = VideoFeatureExtractor(pretrained=True)
    if feature_extractor.model is None: # Check if model loaded successfully
        logging.error("Video feature extractor model failed to initialize. Exiting.")
        return None, None, None

    print("\nExtracting features...")
    print("Extracting training features...")
    train_features, train_combined_features, train_labels, train_paths = extract_features(
        feature_extractor, train_loader, device
    )

    if train_features.size == 0: # Check if feature extraction yielded any results
        logging.error("Training feature extraction failed or produced no features. Cannot proceed.")
        return None, None, None

    if test_loader:
        print("Extracting test features...")
        test_features, test_combined_features, test_labels, test_paths = extract_features(
            feature_extractor, test_loader, device
        )
        if test_features.size == 0 and len(test_files) > 0:
             logging.warning("Test feature extraction produced no features, though test files exist.")
    else: # No test data
        test_features, test_combined_features, test_labels, test_paths = np.array([]), np.array([]), np.array([]), []

    
    print(f"Training features shape: {train_features.shape}")
    if test_features.size > 0: print(f"Test features shape: {test_features.shape}")
    else: print("No test features extracted (no test data or extraction failed).")
    
    print("\nFeature analysis (training features)...")
    if train_features.size > 0 :
        print(f"Mean: {np.mean(train_features):.4f}, Std: {np.std(train_features):.4f}")
        print(f"Min: {np.min(train_features):.4f}, Max: {np.max(train_features):.4f}")
        
        nan_count = np.sum(np.isnan(train_features))
        inf_count = np.sum(np.isinf(train_features))
        if nan_count > 0 or inf_count > 0:
            logging.warning(f"Found {nan_count} NaN and {inf_count} infinite values in training features. Cleaning...")
            train_features = np.nan_to_num(train_features, nan=0.0, posinf=1.0, neginf=-1.0) # Replace with robust values
        if test_features.size > 0:
            test_nan_count = np.sum(np.isnan(test_features))
            test_inf_count = np.sum(np.isinf(test_features))
            if test_nan_count > 0 or test_inf_count > 0:
                 logging.warning(f"Found {test_nan_count} NaN and {test_inf_count} infinite values in test features. Cleaning...")
                 test_features = np.nan_to_num(test_features, nan=0.0, posinf=1.0, neginf=-1.0)

    else:
        logging.warning("No training features to analyze.")

    print("\nDimensionality reduction on training features...")
    if train_features.size > 0 and len(np.unique(train_labels)) > 0 : # Need data and labels for DR plots
        dr_results = perform_dimensionality_reduction(train_features, train_labels, class_labels_encoded_names)
    else:
        logging.warning("Skipping dimensionality reduction due to no training features or labels.")
        dr_results = {}
    
    cv_results = {}
    if len(class_counts) > 1 and train_features.size > 0 and len(np.unique(train_labels)) > 1 : # CV needs >1 class
        print("\nCross-validation...")
        # Pass the class of the model, not an instance
        cv_results, cv_predictions, cv_true_labels = perform_cross_validation(
            train_features, train_labels, EnhancedEnsembleClassifier, 
            cv_folds=CONFIG['cv_folds'],
            label_encoder_classes=class_labels_encoded_names # Pass actual class names
        )
        
        if cv_results:
            print("Cross-validation results (mean/std over folds):")
            for metric, value in cv_results.items():
                if isinstance(value, dict): # For 'overall_metrics_from_folds'
                    print(f"  {metric}:")
                    for k,v in value.items():
                        if not isinstance(v, (dict, np.ndarray)): # print simple values
                             print(f"    {k}: {v:.4f}")
                elif isinstance(value, (float, np.float32, np.float64)):
                    print(f"  {metric}: {value:.4f}")
    else:
        logging.warning("Skipping cross-validation: requires multiple classes and training data.")

    print("\nTraining final model...")
    final_model = EnhancedEnsembleClassifier(random_state=CONFIG['seed'])
    if train_features.size > 0:
        final_model.fit(train_features, train_labels, 
                        use_smote=CONFIG['use_smote'], 
                        feature_selection=CONFIG['feature_selection'])
    else:
        logging.error("Cannot train final model: no training features available.")
        return None, None, None # Critical failure

    test_metrics = {}
    if test_features.size > 0 and len(test_labels) > 0 and final_model.classifiers: # Ensure model was trained and test data exists
        print("\nMaking predictions on test set...")
        test_predictions, test_probabilities, individual_test_predictions, _ = final_model.predict(test_features)
        
        if test_predictions.size > 0:
            print("\nEvaluating model on test set...")
            test_metrics = AdvancedMetrics.compute_all_metrics(
                test_labels, test_predictions, test_probabilities, 
                class_names=class_labels_encoded_names # Pass actual class names
            )
            
            print("\nTest Results:")
            for metric_name, value in test_metrics.items():
                if isinstance(value, (float, np.float32, np.float64)): # Print scalar metrics
                    print(f"{metric_name.replace('_', ' ').title()}: {value:.4f}")
            
            if 'roc_auc' in test_metrics and test_metrics['roc_auc'] is not None:
                print(f"ROC AUC: {test_metrics['roc_auc']:.4f}")
            if 'roc_auc_ovr' in test_metrics and test_metrics['roc_auc_ovr'] is not None:
                print(f"ROC AUC (OvR): {test_metrics['roc_auc_ovr']:.4f}")

            print("\nPlotting confusion matrix for test set...")
            if 'confusion_matrix' in test_metrics and test_metrics['confusion_matrix'].size > 0:
                 AdvancedMetrics.plot_confusion_matrix(test_metrics['confusion_matrix'], class_labels_encoded_names)
            else:
                 logging.warning("No confusion matrix to plot for test set.")

            print("\nIndividual classifier performance on test set:")
            for clf_name, pred in individual_test_predictions.items():
                try:
                    if pred.size > 0:
                        acc = accuracy_score(test_labels, pred)
                        f1 = f1_score(test_labels, pred, average='macro', zero_division=0)
                        print(f"  {clf_name}: Accuracy={acc:.4f}, F1 Macro={f1:.4f}")
                    else:
                        print(f"  {clf_name}: No predictions available.")
                except Exception as e_ind_metric:
                    print(f"  {clf_name}: Failed to compute metrics - {e_ind_metric}")
        else:
            logging.warning("No predictions made on test set. Evaluation skipped.")
    else:
        logging.warning("Skipping test set evaluation: No test data, or final model not trained.")
        test_predictions, test_probabilities, individual_test_predictions = np.array([]), np.array([]), {}


    print("\nAnalyzing feature importance from final model...")
    importance_dict = final_model.get_feature_importance()
    if importance_dict and train_features.size > 0: # Need importances and original feature count
        # Define base feature names
        n_visual_expected = feature_extractor.model.head.in_features if hasattr(feature_extractor.model, 'head') and hasattr(feature_extractor.model.head, 'in_features') else 2048 # Example, should be dynamic
        
        # If DataParallel, access underlying model
        actual_feature_extractor_model = feature_extractor.module if isinstance(feature_extractor, nn.DataParallel) else feature_extractor
        
        # Try to get the visual feature dimension more robustly
        try:
            # Assuming r2plus1d_r50 gives features from before the final projection in the head
            # The input to head.projection is usually the visual feature dim.
            if hasattr(actual_feature_extractor_model.model, 'head') and \
               hasattr(actual_feature_extractor_model.model.head, 'pool') and \
               hasattr(actual_feature_extractor_model.model.head.pool, 'output_size'): #This is complex, direct attribute is better
                 # A common pattern is that the layer before 'projection' or 'fc' has 'out_features' if linear, or it's known.
                 # For R(2+1)D from pytorchvideo, it's often 2048 for r50.
                 # Let's assume visual_features part of train_features has this size.
                 n_visual_actual = train_features.shape[1] - train_combined_features.shape[1]
            else: # Fallback
                 n_visual_actual = train_features.shape[1] - train_combined_features.shape[1] # Infer from extracted shapes
                 if n_visual_actual <=0: n_visual_actual = 2048 # Default if inference fails

        except Exception:
            n_visual_actual = 2048 # Default if introspection fails

        base_feature_names = [f'Visual_{i}' for i in range(n_visual_actual)]
        base_feature_names.extend([f'Temporal_{i}' for i in range(25)]) # Assuming 25 temporal features
        base_feature_names.extend([f'Motion_{i}' for i in range(8)])   # Assuming 8 motion features
        base_feature_names.extend([f'Texture_{i}' for i in range(10)]) # Assuming 10 texture features
        
        current_feature_names_for_plot = base_feature_names
        if final_model.feature_selector and hasattr(final_model.feature_selector, 'get_support'):
            try:
                selected_indices = final_model.feature_selector.get_support(indices=True)
                # Ensure selected_indices are valid for base_feature_names
                if max(selected_indices, default=-1) < len(base_feature_names):
                    current_feature_names_for_plot = [base_feature_names[i] for i in selected_indices]
                else:
                    logging.warning("Feature selector indices out of bounds for base feature names. Using unselected names for plot.")
                    # For the plot, the importance scores are for the *selected* features. So names should match.
                    # If SelectKBest was used, importance scores array length matches k.
                    # We need names for these k features.
                    # The importance_dict values are already aligned with the selected features.
                    # So current_feature_names_for_plot should be the names of the *selected* features.
                    k_selected = len(next(iter(importance_dict.values()), [])) # Get length of one importance array
                    if k_selected == len(selected_indices): # Check consistency
                         current_feature_names_for_plot = [base_feature_names[i] for i in selected_indices]
                    else: # Fallback if there's a mismatch
                        logging.warning("Mismatch in k from SelectKBest and importance scores length. Using generic names for plot.")
                        first_importance_arr = next(iter(importance_dict.values()), [])
                        current_feature_names_for_plot = [f'SelectedFeature_{i}' for i in range(len(first_importance_arr))]

            except Exception as e_fs_plot:
                logging.warning(f"Error getting selected feature names for plot: {e_fs_plot}. Using generic names.")
                first_importance_arr = next(iter(importance_dict.values()), [])
                current_feature_names_for_plot = [f'SelectedFeature_{i}' for i in range(len(first_importance_arr))]
        else: # No feature selection, or selector not available
            # Check if importance_dict features match base_feature_names length
            first_importance_arr = next(iter(importance_dict.values()), [])
            if len(first_importance_arr) != len(base_feature_names):
                logging.warning(f"Mismatch in length of importance scores ({len(first_importance_arr)}) and base_feature_names ({len(base_feature_names)}). Using generic names for plot.")
                current_feature_names_for_plot = [f'Feature_{i}' for i in range(len(first_importance_arr))]


        AdvancedMetrics.plot_feature_importance(importance_dict, current_feature_names_for_plot)
    else:
        logging.info("No feature importance to plot (no training features or model has no importance data).")
    
    print("\nSaving results...")
    results_dict = {
        'config': CONFIG,
        'class_labels': class_labels_encoded_names, # Actual string names of classes
        'test_metrics': test_metrics if test_metrics else "N/A (No test data or evaluation)",
        'cv_results': cv_results if cv_results else "N/A (Cross-validation skipped or failed)",
        'test_predictions': test_predictions.tolist() if test_predictions.size > 0 else "N/A",
        'test_true_labels': test_labels.tolist() if test_labels.size > 0 else "N/A",
        'test_video_paths': test_paths if test_paths else "N/A",
        'individual_test_predictions': {k: v.tolist() for k, v in individual_test_predictions.items()} if individual_test_predictions else "N/A"
    }
    
    os.makedirs(CONFIG['output_folder'], exist_ok=True) # Ensure output folder exists
    results_file_path = os.path.join(CONFIG['output_folder'], 'classification_results.json')
    try:
        with open(results_file_path, 'w') as f:
            # Custom encoder for numpy types if any slip through (e.g. in complex metrics)
            class NumpyEncoder(json.JSONEncoder):
                def default(self, obj):
                    if isinstance(obj, np.integer): return int(obj)
                    if isinstance(obj, np.floating): return float(obj)
                    if isinstance(obj, np.ndarray): return obj.tolist()
                    if isinstance(obj, (np.bool_, bool)): return bool(obj) # Handle numpy bool
                    return super(NumpyEncoder, self).default(obj)
            json.dump(results_dict, f, indent=4, cls=NumpyEncoder)
        print(f"Results saved to: {results_file_path}")
    except Exception as e_json:
        logging.error(f"Failed to save results to JSON: {e_json}")

    print("\n" + "="*60)
    print("FINAL SUMMARY")
    print("="*60)
    print(f"Dataset: {len(video_files)} videos, {len(class_labels_encoded_names)} classes ({', '.join(class_labels_encoded_names)})")
    
    if test_metrics and isinstance(test_metrics, dict): # Check if test_metrics is a dict and not "N/A"
        print(f"Best Test Accuracy: {test_metrics.get('accuracy', 0.0):.4f}")
        print(f"Best Test F1 (Macro): {test_metrics.get('f1_macro', 0.0):.4f}")
        print(f"Best Test Balanced Accuracy: {test_metrics.get('balanced_accuracy', 0.0):.4f}")
    else:
        print("Test metrics: N/A")
        
    if cv_results and isinstance(cv_results, dict):
        print(f"CV Accuracy (Mean): {cv_results.get('accuracy_mean', 0.0):.4f} ± {cv_results.get('accuracy_std', 0.0):.4f}")
        print(f"CV F1 (Macro Mean): {cv_results.get('f1_macro_mean', 0.0):.4f} ± {cv_results.get('f1_macro_std', 0.0):.4f}")
        print(f"CV Bal Acc (Mean): {cv_results.get('balanced_accuracy_mean', 0.0):.4f} ± {cv_results.get('balanced_accuracy_std', 0.0):.4f}")
    else:
        print("Cross-validation results: N/A")
    
    print("="*60)
    
    return final_model, test_metrics, results_dict

def safe_main():
    try:
        return main()
    except KeyboardInterrupt:
        logging.error("Process interrupted by user.")
        return None, None, None
    except Exception as e:
        logging.error(f"A fatal error occurred in main execution: {e}")
        import traceback
        traceback.print_exc() # Print full traceback for debugging
        return None, None, None

if __name__ == "__main__":
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            print(f"CUDA available: {torch.cuda.get_device_name(0)}") # Specify device 0
            print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
        except Exception as e:
            print(f"Could not get CUDA device info: {e}")
    else:
        print("CUDA not available. Running on CPU.")
    
    final_model, run_metrics, run_results_dict = safe_main()
    
    if final_model and run_metrics: # Check if model and metrics are valid
        print("\n✅ Pipeline completed successfully!")
    else:
        print("\n❌ Pipeline failed or critical data was missing. Check logs for details.")